# 30 — Multi-NR LGBM: Morgan+RDKit + ESM-2 Protein Embeddings

Train LGBM on combined PXR CRC + ChEMBL NR data where each (compound, target) pair
is featurized as `[compound_features | protein_embedding]`.

- Compound features: Morgan (2048) + RDKit (217) = 2265-dim
- Protein features: ESM-2 global embedding (320-dim)
- Total input: **2585 features**

The protein embedding allows the model to learn target-specific SAR patterns
from all 7 NR targets simultaneously.

In [1]:
import sys, warnings, json
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.featurize import combined, impute
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5

LGBM_PARAMS = dict(
    n_estimators=1200, num_leaves=64, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.2,
    min_child_samples=10, n_jobs=4, verbose=-1
)

TARGET_WEIGHTS = {
    'PXR':   1.0,
    'VDR':   0.50,
    'FXR':   0.30,
    'LXRa':  0.25,
    'RXRa':  0.25,
    'PPARg': 0.15,
    'PPARa': 0.15,
}

print('Setup complete.')

Setup complete.


## 1. Load data and protein embeddings

In [2]:
train = load_train()
te    = load_test()
nr    = pd.read_parquet('../data/external/chembl_nr_targets.parquet')

# Load ESM-2 protein embeddings
esm2_arr   = np.load(DATA_PROCESSED / 'nr_esm2_embeddings.npy')   # (n_proteins, 320)
with open(DATA_PROCESSED / 'nr_esm2_names.json') as f:
    esm2_names = json.load(f)

# Build lookup: target_name -> embedding vector
esm2_lookup = {name: esm2_arr[i] for i, name in enumerate(esm2_names)}
PROTEIN_DIM = esm2_arr.shape[1]  # 320

print(f'CRC train: {len(train):,}  |  ChEMBL NR: {len(nr):,}  |  Test: {len(te):,}')
print(f'ESM-2 embedding dim: {PROTEIN_DIM}')
print(f'Available targets: {esm2_names}')
print(nr['target_name'].value_counts())

CRC train: 4,139  |  ChEMBL NR: 11,511  |  Test: 513
ESM-2 embedding dim: 320
Available targets: ['PXR', 'VDR', 'FXR', 'LXRa', 'RXRa', 'PPARg', 'PPARa']
target_name
PPARg    4311
FXR      3187
RXRa     1364
LXRa     1175
PXR       947
VDR       523
PPARa       4
Name: count, dtype: int64


## 2. Featurize compounds

In [3]:
# Filter ChEMBL NR: pEC50 quality range
nr_filt = nr[(nr['pec50'] >= 3.0) & (nr['pec50'] <= 10.0)].copy()
nr_filt = nr_filt[nr_filt['target_name'].isin(esm2_lookup)].copy()
nr_filt['weight'] = nr_filt['target_name'].map(TARGET_WEIGHTS).fillna(0.1)
print(f'ChEMBL NR after quality filter: {len(nr_filt):,}')

print('Featurizing CRC train...')
X_tr_chem = impute(combined(train['smiles'].tolist()))   # (4139, 2265)
y_tr = train['pec50'].values

print('Featurizing ChEMBL NR...')
nr_smiles = nr_filt['smiles'].tolist()
X_nr_chem_raw = combined(nr_smiles)  # may have NaN rows for bad SMILES

# Mask out rows with invalid SMILES (all-NaN after featurization)
valid_nr_mask = ~np.all(np.isnan(X_nr_chem_raw), axis=1)
X_nr_chem_raw = X_nr_chem_raw[valid_nr_mask]
nr_filt_valid = nr_filt.iloc[valid_nr_mask].reset_index(drop=True)
X_nr_chem = impute(X_nr_chem_raw)
print(f'  Valid ChEMBL NR compounds: {len(nr_filt_valid):,} / {len(nr_filt):,}')

print('Featurizing test...')
X_te_chem = impute(combined(te['smiles'].tolist()))     # (513, 2265)

CHEM_DIM = X_tr_chem.shape[1]  # 2265
TOTAL_DIM = CHEM_DIM + PROTEIN_DIM  # 2585
print(f'\nChem features: {CHEM_DIM}  |  Protein features: {PROTEIN_DIM}  |  Total: {TOTAL_DIM}')

ChEMBL NR after quality filter: 11,496
Featurizing CRC train...


Featurizing ChEMBL NR...


  Valid ChEMBL NR compounds: 11,496 / 11,496
Featurizing test...



Chem features: 2265  |  Protein features: 320  |  Total: 2585


## 3. Build multi-NR training matrix

Concatenate `[compound_features | protein_embedding]` for each (compound, target) row.

In [4]:
pxr_emb = esm2_lookup['PXR']  # (320,)

# CRC train: append PXR protein embedding to each compound
pxr_emb_tile_tr = np.tile(pxr_emb, (len(X_tr_chem), 1))  # (4139, 320)
X_tr_full = np.hstack([X_tr_chem, pxr_emb_tile_tr])       # (4139, 2585)
w_tr = np.ones(len(y_tr))  # weight=1.0 for PXR CRC

# ChEMBL NR: append per-target protein embedding
nr_protein_embs = np.stack(
    [esm2_lookup[t] for t in nr_filt_valid['target_name']], axis=0
)  # (n_nr, 320)
X_nr_full = np.hstack([X_nr_chem, nr_protein_embs])        # (n_nr, 2585)
y_nr = nr_filt_valid['pec50'].values
w_nr = nr_filt_valid['weight'].values

# Test: append PXR protein embedding
pxr_emb_tile_te = np.tile(pxr_emb, (len(X_te_chem), 1))  # (513, 320)
X_te_full = np.hstack([X_te_chem, pxr_emb_tile_te])       # (513, 2585)

print(f'Train (CRC): {X_tr_full.shape}')
print(f'NR augment:  {X_nr_full.shape}')
print(f'Test:        {X_te_full.shape}')

Train (CRC): (4139, 2585)
NR augment:  (11496, 2585)
Test:        (513, 2585)


## 4. Scaffold 5-fold CV (evaluated on CRC train only)

In [5]:
scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    # Combine CRC fold-train with all ChEMBL NR data
    X_fold = np.vstack([X_tr_full[tr_idx], X_nr_full])
    y_fold = np.concatenate([y_tr[tr_idx], y_nr])
    w_fold = np.concatenate([w_tr[tr_idx], w_nr])

    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_fold, y_fold, sample_weight=w_fold)

    oof[va_idx] = m.predict(X_tr_full[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    fold_metrics.append(met)
    print(f'  Fold {fold_i+1}: RAE={met["RAE"]:.4f}  Spearman={met["Spearman"]:.4f}')

oof_rae = rae_fn(y_tr, oof)
cv_df = pd.DataFrame(fold_metrics)
print(f'\nOOF RAE (global): {oof_rae:.4f}')
print(f'Mean fold RAE:    {cv_df["RAE"].mean():.4f} +/- {cv_df["RAE"].std():.4f}')
print(f'Mean Spearman:    {cv_df["Spearman"].mean():.4f}')
print(f'\nBaseline comparisons:')
print(f'  LGBM (PXR only, combined):    ~0.575')
print(f'  NR-weighted LGBM (no prot):   see nb27')
print(f'  Morgan+ESM-2 multi-NR (this): {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_morgan_esm2_nr.npy', oof)

  Fold 1: RAE=0.5174  Spearman=0.7814


  Fold 2: RAE=0.5989  Spearman=0.6810


  Fold 3: RAE=0.6082  Spearman=0.6870


  Fold 4: RAE=0.5744  Spearman=0.6939


  Fold 5: RAE=0.6061  Spearman=0.6881

OOF RAE (global): 0.5763
Mean fold RAE:    0.5810 +/- 0.0380
Mean Spearman:    0.7063

Baseline comparisons:
  LGBM (PXR only, combined):    ~0.575
  NR-weighted LGBM (no prot):   see nb27
  Morgan+ESM-2 multi-NR (this): 0.5763


## 5. Full retrain on all data + test predictions

In [6]:
X_full = np.vstack([X_tr_full, X_nr_full])
y_full = np.concatenate([y_tr, y_nr])
w_full = np.concatenate([w_tr, w_nr])

final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_full, y_full, sample_weight=w_full)

te_preds = np.clip(final_m.predict(X_te_full), y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_morgan_esm2_nr.npy', te_preds)

print(f'Test pEC50 predictions:')
print(f'  min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}')

Test pEC50 predictions:
  min=2.42  median=4.94  max=6.03


## 6. Save submission

In [7]:
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all(), 'Submission validation failed'

out_path = SUBMISSIONS / '30_morgan_esm2_multinr.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'OOF RAE: {oof_rae:.4f}')
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\30_morgan_esm2_multinr.csv
OOF RAE: 0.5763
count    513.000
mean       4.817
std        0.653
min        2.419
25%        4.444
50%        4.944
75%        5.298
max        6.027
Name: pEC50, dtype: float64
